# Qwen3.5-27B + SGLang + OpenAI Client Quickstart

この Notebook は、Docker で SGLang API サーバを起動したあとに、Python の OpenAI ライブラリから Qwen3.5-27B を使う最短手順をまとめたものです。


## 1. 前提
- Docker が使える
- NVIDIA GPU が使える
- `HF_TOKEN` が必要に応じて設定されている


In [1]:
!docker ps --filter name=qwen35-sglang-api
!curl -s http://127.0.0.1:30000/v1/models || true


CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


## 2. OpenAI クライアント初期化


In [2]:
import os
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:30000/v1')
client = OpenAI(
    api_key='EMPTY',
    base_url=BASE_URL,
)
print('Using base_url =', BASE_URL)
client


Using base_url = http://127.0.0.1:30009/v1


## 3. テキスト推論


In [3]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'user', 'content': 'SGLangとは何かを日本語で2文で説明してください。'}
    ],
    max_tokens=64,
)
resp


ChatCompletion(id='8894f6c96fee4aef821db0f4583531b3', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: SGLang (a software library/framework for large language models).\n    *   Language: Japanese.\n    *   Constraint: Exactly 2 sentences (2 文).\n\n2.  **Identify Key Information'), matched_stop=None)], created=1774936958, model='Qwen/Qwen3.5-27B', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=64, prompt_tokens=24, total_tokens=88, completion_tokens_details=None, prompt_tokens_details=None, reasoning_tokens=0), metadata={'weight_version': 'default'})

## 4. 画像入力推論


In [4]:
import base64
import mimetypes
from pathlib import Path

img = Path('image.png')  # ここを自分の画像に置き換える
if img.exists():
    mime = mimetypes.guess_type(img.name)[0] or 'image/png'
    image_url = 'data:' + mime + ';base64,' + base64.b64encode(img.read_bytes()).decode('utf-8')

    resp = client.chat.completions.create(
        model='Qwen/Qwen3.5-27B',
        messages=[
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': 'この画像の内容を説明してください。'},
                    {'type': 'image_url', 'image_url': {'url': image_url}},
                ],
            }
        ],
        max_tokens=64,
    )
    resp
else:
    print('image.png が見つからないため、このセルはスキップされました。')


image.png が見つからないため、このセルはスキップされました。


## 5. 補足
Qwen3.5 は thinking mode が既定のため、レスポンスは `reasoning_content` 側に現れることがあります。
